In [12]:
!pip install pandas

In [13]:
# Internal payments data (what the company recorded)
internal_data = StringIO("""transaction_id,amount,date,recipient
TXN001,5000.00,2026-04-01,ABC Logistics
TXN002,12000.00,2026-04-02,XYZ Supplies
TXN003,7500.00,2026-04-03,ABC Logistics
TXN004,3000.00,2026-04-05,MNO Services
TXN005,20000.00,2026-04-05,DEF Trading
""")

internal_df = pd.read_csv(internal_data)

print("Internal payments table:")
print(internal_df)

Internal payments table:
  transaction_id   amount        date      recipient
0         TXN001   5000.0  2026-04-01  ABC Logistics
1         TXN002  12000.0  2026-04-02   XYZ Supplies
2         TXN003   7500.0  2026-04-03  ABC Logistics
3         TXN004   3000.0  2026-04-05   MNO Services
4         TXN005  20000.0  2026-04-05    DEF Trading


In [14]:
# Bank statement data (what the bank actually processed)
bank_data = StringIO("""transaction_id,amount,date,description
TXN001,5000.00,2026-04-01,Payment to ABC Logistics
TXN002,11800.00,2026-04-02,Payment to XYZ Supplies
TXN004,3000.00,2026-04-05,Payment to MNO Services
TXN005,20000.00,2026-04-05,Payment to DEF Trading
TXN006,4500.00,2026-04-06,Payment to Unknown Vendor
""")

bank_df = pd.read_csv(bank_data)

print("Bank statement table:")
print(bank_df)

Bank statement table:
  transaction_id   amount        date                description
0         TXN001   5000.0  2026-04-01   Payment to ABC Logistics
1         TXN002  11800.0  2026-04-02    Payment to XYZ Supplies
2         TXN004   3000.0  2026-04-05    Payment to MNO Services
3         TXN005  20000.0  2026-04-05     Payment to DEF Trading
4         TXN006   4500.0  2026-04-06  Payment to Unknown Vendor


In [15]:
# Merge on transaction_id, keeping all rows from both tables (outer join)
merged = pd.merge(internal_df, bank_df, on='transaction_id', how='outer', suffixes=('_internal', '_bank'))

print("Merged table (before adding status):")
print(merged)

Merged table (before adding status):
  transaction_id  amount_internal date_internal      recipient  amount_bank  \
0         TXN001           5000.0    2026-04-01  ABC Logistics       5000.0   
1         TXN002          12000.0    2026-04-02   XYZ Supplies      11800.0   
2         TXN003           7500.0    2026-04-03  ABC Logistics          NaN   
3         TXN004           3000.0    2026-04-05   MNO Services       3000.0   
4         TXN005          20000.0    2026-04-05    DEF Trading      20000.0   
5         TXN006              NaN           NaN            NaN       4500.0   

    date_bank                description  
0  2026-04-01   Payment to ABC Logistics  
1  2026-04-02    Payment to XYZ Supplies  
2         NaN                        NaN  
3  2026-04-05    Payment to MNO Services  
4  2026-04-05     Payment to DEF Trading  
5  2026-04-06  Payment to Unknown Vendor  


In [16]:
# Define a function that decides the status based on the row data
def reconciliation_status(row):
    # Case 1: Only bank side exists (extra in bank)
    if pd.isna(row['amount_internal']) and pd.notna(row['amount_bank']):
        return 'Bank only (no internal record)'
    # Case 2: Only internal side exists (missing from bank)
    elif pd.notna(row['amount_internal']) and pd.isna(row['amount_bank']):
        return 'Internal only (missing from bank)'
    # Case 3: Both exist and amounts are equal
    elif row['amount_internal'] == row['amount_bank']:
        return 'Matched'
    # Case 4: Both exist but amounts differ
    else:
        return 'Amount mismatch'

In [17]:
merged['status'] = merged.apply(reconciliation_status, axis=1)

print("Merged table with status:")
print(merged[['transaction_id', 'amount_internal', 'amount_bank', 'status']])

Merged table with status:
  transaction_id  amount_internal  amount_bank  \
0         TXN001           5000.0       5000.0   
1         TXN002          12000.0      11800.0   
2         TXN003           7500.0          NaN   
3         TXN004           3000.0       3000.0   
4         TXN005          20000.0      20000.0   
5         TXN006              NaN       4500.0   

                              status  
0                            Matched  
1                    Amount mismatch  
2  Internal only (missing from bank)  
3                            Matched  
4                            Matched  
5     Bank only (no internal record)  


In [18]:
# Calculate difference: bank amount minus internal amount
merged['difference'] = merged['amount_bank'] - merged['amount_internal']

# Show the final report with all columns
report = merged[['transaction_id', 'amount_internal', 'amount_bank', 'difference', 'status']]

print("\n=== FINAL RECONCILIATION REPORT ===")
print(report.to_string(index=False))


=== FINAL RECONCILIATION REPORT ===
transaction_id  amount_internal  amount_bank  difference                            status
        TXN001           5000.0       5000.0         0.0                           Matched
        TXN002          12000.0      11800.0      -200.0                   Amount mismatch
        TXN003           7500.0          NaN         NaN Internal only (missing from bank)
        TXN004           3000.0       3000.0         0.0                           Matched
        TXN005          20000.0      20000.0         0.0                           Matched
        TXN006              NaN       4500.0         NaN    Bank only (no internal record)


In [19]:
print("\n=== SUMMARY FOR TREASURY MANAGER ===\n")

print(f"Total internal payments recorded: {len(internal_df)}")
print(f"Total bank entries processed: {len(bank_df)}")

matched_count = len(report[report['status'] == 'Matched'])
mismatch_count = len(report[report['status'] == 'Amount mismatch'])
missing_count = len(report[report['status'] == "Internal only (missing from bank)"])
extra_count = len(report[report['status'] == "Bank only (no internal record)"])

print(f"\n✅ Matched: {matched_count}")
print(f"⚠️  Amount mismatches: {mismatch_count}")
print(f"❌ Missing from bank (internal not paid): {missing_count}")
print(f"❓ Extra in bank (bank error or fraud?): {extra_count}")

if missing_count > 0:
    print("\n🚨 Action required: Investigate why these payments didn't hit the bank.")
if extra_count > 0:
    print("🚨 Action required: Verify if these extra bank entries are legitimate.")


=== SUMMARY FOR TREASURY MANAGER ===

Total internal payments recorded: 5
Total bank entries processed: 5

✅ Matched: 3
⚠️  Amount mismatches: 1
❌ Missing from bank (internal not paid): 1
❓ Extra in bank (bank error or fraud?): 1

🚨 Action required: Investigate why these payments didn't hit the bank.
🚨 Action required: Verify if these extra bank entries are legitimate.


In [22]:
from datetime import timedelta

def is_date_within_tolerance(date1, date2, max_days=1):
    """
    Returns True if two dates are within max_days of each other.
    Handles missing dates (NaN) gracefully.
    """
    if pd.isna(date1) or pd.isna(date2):
        return False
    diff = abs(pd.to_datetime(date1) - pd.to_datetime(date2))
    return diff <= timedelta(days=max_days)

In [23]:
# First, do the exact merge on transaction_id (same as before)
exact_merge = pd.merge(internal_df, bank_df, on='transaction_id', how='outer', suffixes=('_int', '_bank'))

# Mark which rows already matched exactly (both sides present and amounts equal)
exact_merge['exact_match'] = (exact_merge['amount_int'] == exact_merge['amount_bank']) & \
                              pd.notna(exact_merge['amount_int']) & \
                              pd.notna(exact_merge['amount_bank'])

print("After exact merge (with flag):")
print(exact_merge[['transaction_id', 'amount_int', 'amount_bank', 'exact_match']])

After exact merge (with flag):
  transaction_id  amount_int  amount_bank  exact_match
0         TXN001      5000.0       5000.0         True
1         TXN002     12000.0      11800.0        False
2         TXN003      7500.0          NaN        False
3         TXN004      3000.0       3000.0         True
4         TXN005     20000.0      20000.0         True
5         TXN006         NaN       4500.0        False


In [28]:
# Separate the rows that are unmatched from internal side
unmatched_internal = exact_merge[pd.isna(exact_merge['amount_bank']) | (exact_merge['exact_match'] == False)]

# Separate the rows that are unmatched from bank side
unmatched_bank = exact_merge[pd.isna(exact_merge['amount_int']) | (exact_merge['exact_match'] == False)]

print(f"Unmatched internal rows: {len(unmatched_internal)}")
print(f"Unmatched bank rows: {len(unmatched_bank)}")

Unmatched internal rows: 3
Unmatched bank rows: 3


In [30]:
fuzzy_matches = []

# For each internal row that didn't match exactly
for idx_int, row_int in unmatched_internal.iterrows():
    if pd.isna(row_int['amount_int']):
        continue  # skip if no internal amount (this is bank-only row)

    # Search in unmatched_bank for a potential match
    for idx_bank, row_bank in unmatched_bank.iterrows():
        if pd.isna(row_bank['amount_bank']):
            continue

        # Condition 1: amounts are equal
        amount_ok = (row_int['amount_int'] == row_bank['amount_bank'])

        # Condition 2: recipient name appears in bank description (simple)
        recipient = str(row_int['recipient']).lower()
        description = str(row_bank['description']).lower()
        recipient_ok = recipient in description or description in recipient

        # Condition 3: dates within 1 day
        date_ok = is_date_within_tolerance(row_int['date_int'], row_bank['date_bank'], max_days=1)

        if amount_ok and recipient_ok and date_ok:
            fuzzy_matches.append({
                'transaction_id_int': row_int['transaction_id'],
                'transaction_id_bank': row_bank['transaction_id'],
                'amount': row_int['amount_int'],
                'date_int': row_int['date_int'],
                'date_bank': row_bank['date_bank'],
                'status': 'Fuzzy matched (amount+recipient+date tolerance)'
            })
            # Remove this bank row from consideration (to avoid double matching)
            unmatched_bank = unmatched_bank.drop(idx_bank)
            break  # stop searching for this internal row

# Convert to DataFrame for easy viewing
if fuzzy_matches:
    fuzzy_df = pd.DataFrame(fuzzy_matches)
    print("Fuzzy matches found:")
    print(fuzzy_df)
else:
    print("No fuzzy matches found with current data.")

No fuzzy matches found with current data.


In [31]:
# Start with the exact matches (where exact_match is True)
exact_matched = exact_merge[exact_merge['exact_match'] == True].copy()
exact_matched['final_status'] = 'Matched (exact)'

# Create a DataFrame for fuzzy matches (if any)
if fuzzy_matches:
    fuzzy_matched = pd.DataFrame(fuzzy_matches)
    fuzzy_matched['final_status'] = fuzzy_matched['status']
    # Keep only columns we care about for final report
    fuzzy_matched = fuzzy_matched[['transaction_id_int', 'amount', 'date_int', 'date_bank', 'final_status']]
    fuzzy_matched.rename(columns={'transaction_id_int': 'transaction_id', 'amount': 'amount_internal', 'date_int': 'date_internal'}, inplace=True)
else:
    fuzzy_matched = pd.DataFrame()

# Remaining unmatched internal rows (still missing from bank)
remaining_internal = unmatched_internal[~unmatched_internal.index.isin([m['transaction_id_int'] for m in fuzzy_matches])]
remaining_internal = remaining_internal[pd.notna(remaining_internal['amount_int'])]
remaining_internal['final_status'] = 'Internal only (missing from bank)'
remaining_internal = remaining_internal[['transaction_id', 'amount_int', 'date_int', 'final_status']]
remaining_internal.rename(columns={'amount_int': 'amount_internal', 'date_int': 'date_internal'}, inplace=True)

# Remaining unmatched bank rows (extra in bank)
remaining_bank = unmatched_bank[pd.notna(unmatched_bank['amount_bank'])]
remaining_bank['final_status'] = 'Bank only (no internal record)'
remaining_bank = remaining_bank[['transaction_id', 'amount_bank', 'date_bank', 'final_status']]
remaining_bank.rename(columns={'amount_bank': 'amount_internal', 'date_bank': 'date_internal'}, inplace=True)

# Combine all parts
final_report = pd.concat([exact_matched[['transaction_id', 'amount_int', 'date_int', 'final_status']].rename(columns={'amount_int': 'amount_internal', 'date_int': 'date_internal'}),
                          fuzzy_matched,
                          remaining_internal,
                          remaining_bank], ignore_index=True)

print("\n=== FINAL RECONCILIATION REPORT WITH DATE TOLERANCE ===")
print(final_report.to_string(index=False))


=== FINAL RECONCILIATION REPORT WITH DATE TOLERANCE ===
transaction_id  amount_internal date_internal                      final_status
        TXN001           5000.0    2026-04-01                   Matched (exact)
        TXN004           3000.0    2026-04-05                   Matched (exact)
        TXN005          20000.0    2026-04-05                   Matched (exact)
        TXN002          12000.0    2026-04-02 Internal only (missing from bank)
        TXN003           7500.0    2026-04-03 Internal only (missing from bank)
        TXN002          11800.0    2026-04-02    Bank only (no internal record)
        TXN006           4500.0    2026-04-06    Bank only (no internal record)


/tmp/ipykernel_44359/1313945655.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  remaining_bank['final_status'] = 'Bank only (no internal record)'


In [32]:
# Save the report to a CSV file in the Colab environment
report.to_csv('reconciliation_report.csv', index=False)

print("✅ Report saved as 'reconciliation_report.csv'")

✅ Report saved as 'reconciliation_report.csv'
